In [ ]:
# ============================================================
# NB_TRNSF_GoldVentas_DimProducto
# Capa: Gold | Modelo: ventas | Tabla: d_producto
# Fuente: Ih_silver.ADVWDB.Product
# Carga total — overwrite
# Todo desde config — sin hardcodeos
# ============================================================
from pyspark.sql import functions as F

# --- IDs técnicos (inmutables del TP) ---
# ============================================================
# IDs de workspace y lakehouses — SIN hardcodear (TP)
# ============================================================
# lakehouse.get() funciona en pipeline y en ejecución manual.
# El pipeline setea lh_gold como defaultLakehouse de los notebooks Gold.
_lh_gold   = notebookutils.lakehouse.get('lh_gold')
_lh_silver = notebookutils.lakehouse.get('lh_silver')
WS_ID      = _lh_gold.workspaceId
ART_GOLD   = _lh_gold.id
ART_SILVER = _lh_silver.id

# Leer config Silver — construye path origen dinámicamente
config_prod = spark.sql("""
    SELECT origen, nombre_tabla
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen = 'ADVWDB' AND nombre_tabla = 'Product' AND activo = 1
    LIMIT 1
""").collect()
if not config_prod:
    raise Exception("Config no encontrada para ADVWDB/Product en origin_bronze_silver")
_origen_prod = config_prod[0]['origen']
_tabla_prod  = config_prod[0]['nombre_tabla']

# FIX D01/D02: leer modelo y nombre_tabla Gold desde origin_gold
_og_prod = spark.sql("""
    SELECT modelo, nombre_tabla
    FROM lh_config.dbo.origin_gold
    WHERE nombre_notebook = 'NB_TRNSF_GoldVentas_DimProducto' AND activo = 1
    LIMIT 1
""").collect()
if not _og_prod:
    raise Exception("Config no encontrada para DimProducto en origin_gold")
_modelo_prod      = _og_prod[0]['modelo']       # 'ventas'
_tabla_gold_prod  = _og_prod[0]['nombre_tabla'] # 'd_producto'

abfs_silver = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_SILVER}/Tables/{_origen_prod}/{_tabla_prod}"
)
abfs_gold = (
    f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{ART_GOLD}/Tables/{_modelo_prod}/{_tabla_gold_prod}"
)

print(f"✅ Paths configurados desde config (origin_bronze_silver + origin_gold)")
print(f"   origen Silver : {_origen_prod} | tabla : {_tabla_prod}")
print(f"   modelo Gold   : {_modelo_prod} | tabla : {_tabla_gold_prod}")
print(f"   silver : {abfs_silver}")
print(f"   gold   : {abfs_gold}")


In [ ]:
# ============================================================
# Leer desde Silver y transformar a dimensión
# ============================================================
# Leer via catálogo Spark — más robusto que path ABFS desde notebook Gold
df_silver = spark.sql(f"SELECT * FROM lh_silver.{_origen_prod}.{_tabla_prod} LIMIT 1000")
print(f"✅ Filas Silver Product: {df_silver.count()}")
display(df_silver.limit(5))

if df_silver.count() == 0:
    raise Exception("❌ Silver Product tiene 0 filas — ejecutar DP_INGST_SilverADVWDB primero")


In [ ]:
# ============================================================
# Transformar a dimensión d_producto
# Renombrar + seleccionar columnas relevantes
# ============================================================
df_dim = (df_silver
    .withColumnRenamed("ProductID",         "id_producto")
    .withColumnRenamed("Name",              "nombre_producto")
    .withColumnRenamed("ProductNumber",     "codigo_producto")
    .withColumnRenamed("Color",             "color")
    .withColumnRenamed("StandardCost",      "costo_estandar")
    .withColumnRenamed("ListPrice",         "precio_lista")
    .withColumnRenamed("Size",              "talla")
    .withColumnRenamed("ProductLine",       "linea_producto")
    .withColumnRenamed("Class",             "clase")
    .withColumnRenamed("Style",             "estilo")
    .withColumnRenamed("SellStartDate",     "fecha_inicio_venta")
    .withColumnRenamed("SellEndDate",       "fecha_fin_venta")
    .withColumnRenamed("DiscontinuedDate",  "fecha_discontinuacion")
    .withColumnRenamed("ModifiedDate",      "fecha_modificacion")
    .select(
        "id_producto", "nombre_producto", "codigo_producto",
        "color", "talla", "linea_producto", "clase", "estilo",
        "costo_estandar", "precio_lista",
        "fecha_inicio_venta", "fecha_fin_venta",
        "fecha_discontinuacion", "fecha_modificacion"
    )
)

print(f"✅ Filas dim_producto: {df_dim.count()}")
display(df_dim.select("id_producto","nombre_producto","precio_lista","linea_producto").limit(10))


In [ ]:
# ============================================================
# Guardar en Gold — carga total
# ============================================================
try:
    notebookutils.fs.rm(abfs_gold, recurse=True)
    print(f"🗑️  Tabla anterior eliminada")
except:
    print("ℹ️  Primera carga")

(df_dim.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(abfs_gold))

print(f"✅ d_producto guardada en Gold: {df_dim.count()} filas")


In [ ]:
# ============================================================
# Verificar resultado
# ============================================================
df_ver = spark.read.format("delta").load(abfs_gold)
print(f"✅ Total filas en Gold d_producto: {df_ver.count()}")
display(df_ver
    .select("id_producto","nombre_producto","precio_lista","linea_producto")
    .orderBy("id_producto")
    .limit(10))
